In [8]:
import os
import cv2
import random

FOLDER = "ADL-Rundle-8"

IMG_DIR = f"/home/anhtv/ETC/MOT15/train/{FOLDER}/img1"
GT_FILE = f"/home/anhtv/ETC/MOT15/train/{FOLDER}/gt/gt.txt"

# Lưu tất cả bbox theo frame
gt_data = {}

with open(GT_FILE, "r") as f:
    for line in f:
        frame, obj_id, x, y, w, h, conf, wx, wy, wz = line.strip().split(",")

        frame = int(frame)
        obj_id = int(obj_id)
        x, y, w, h = map(float, [x, y, w, h])

        if frame not in gt_data:
            gt_data[frame] = []

        gt_data[frame].append((obj_id, x, y, w, h))

# random color cho mỗi ID
id_colors = {}

def get_color(obj_id):
    if obj_id not in id_colors:
        id_colors[obj_id] = (
            random.randint(0,255),
            random.randint(0,255),
            random.randint(0,255)
        )
    return id_colors[obj_id]

# Prepare video writer
first_frame = min(gt_data.keys())
first_img_path = os.path.join(IMG_DIR, f"{first_frame:06d}.jpg")
img = cv2.imread(first_img_path)

H, W = img.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("gt_tracking.mp4", fourcc, 30, (img.shape[1], img.shape[0]))

# vẽ từng frame
for frame in sorted(gt_data.keys()):
    img_name = f"{frame:06d}.jpg"
    img_path = os.path.join(IMG_DIR, img_name)

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)

    for obj_id, x, y, w, h in gt_data[frame]:
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + w), int(y + h)

        color = get_color(obj_id)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            img,
            f"ID {obj_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

    out.write(img)
    cv2.imshow("GT Tracking", img)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cv2.destroyAllWindows()
out.release()

print("Video saved as gt_tracking.mp4")

Video saved as gt_tracking.mp4
